In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('../data/track_analysis_results.csv')
print(f"Loaded {len(df)} tracks")
df.head()

Loaded 50 tracks


,filename,genre,bpm,key,camelot,energy,key_confidence
0,../data/genres_original\classical\classical.00...,classical,95.70,D Major,10B,0.098,0.81
1,../data/genres_original\classical\classical.00...,classical,112.35,B Minor,10A,0.079,0.92
2,../data/genres_original\classical\classical.00...,classical,99.38,E Major,12B,0.107,0.70
3,../data/genres_original\classical\classical.00...,classical,71.78,C Major,8B,0.088,0.75
4,../data/genres_original\classical\classical.00...,classical,161.50,A Minor,8A,0.115,0.82


In [4]:
from src.analyzer import camelot_compatibility

In [8]:
def compatibility_score(track1, track2):
    """
    Scores how well track_b would transition from track_a.
    Returns 0-100.
    """

    # --- BPM compatibility (the gate) ---
    diff = np.abs(track1['bpm'] - track2['bpm'])
    bpm_score = 0
    if diff <= 5:
        bpm_score = 100
    elif diff <= 15:
        bpm_score = 80
    elif diff <= 30:
        bpm_score = 50
    else:
        bpm_score = 15

    # --- Camelot key compatibility (the polish) ---
    key_score = camelot_compatibility(track1['camelot'], track2['camelot'])

    # --- Energy flow (the journey) ---
    # Small energy steps are good (gradual build/wind-down)
    # Huge energy jumps in either direction are jarring
    energy_diff = abs(track1['energy'] - track2['energy'])
    energy_score = max(0, 100 - (energy_diff * 250))  # scaled penalty

     # --- Weighted combination ---
    # BPM weighted highest since it's the listening "baseline"
    # Key second — the harmonic polish
    # Energy third — shapes the overall journey
    final_score = (
        0.45 * bpm_score   +
        0.35 * key_score   +
        0.20 * energy_score
    )
    
    return round(final_score, 1)
    
track1 = df.iloc[0].to_dict()
track2 = df.iloc[1].to_dict()

score = compatibility_score(track1, track2)
print(f"{track1['filename']} ({track1['bpm']} BPM, {track1['camelot']})")
print(f"  → {track2['filename']} ({track2['bpm']} BPM, {track2['camelot']})")
print(f"  Compatibility: {score}/100")
    


../data/genres_original\classical\classical.00000.wav (95.7 BPM, 10B)
  → ../data/genres_original\classical\classical.00001.wav (112.35 BPM, 10A)
  Compatibility: 59.0/100


In [9]:
# Find the two most BPM-different classical tracks
classical = df[df['genre'] == 'classical'].sort_values('bpm')
slow = classical.iloc[0].to_dict()
fast = classical.iloc[-1].to_dict()

print("--- Worst case: huge BPM gap ---")
score_bad = compatibility_score(slow, fast)
print(f"{slow['filename']} ({slow['bpm']} BPM, {slow['camelot']})")
print(f"  → {fast['filename']} ({fast['bpm']} BPM, {fast['camelot']})")
print(f"  Compatibility: {score_bad}/100")

print("\n--- Best case: same track ---")
score_perfect = compatibility_score(slow, slow)
print(f"Compatibility with itself: {score_perfect}/100")

--- Worst case: huge BPM gap ---
../data/genres_original\classical\classical.00003.wav (71.78 BPM, 8B)
  → ../data/genres_original\classical\classical.00009.wav (234.91 BPM, 6A)
  Compatibility: 25.7/100

--- Best case: same track ---
Compatibility with itself: 100.0/100


In [15]:
def greedy_reorder(df, lookahead=3):
    remaining = df.to_dict('records')
    remaining.sort(key=lambda x: x['energy'])
    current = remaining.pop(0)
    ordered = [current]
    
    while remaining:
        # Score all candidates against current track
        scored = [(compatibility_score(current, c), i, c) for i, c in enumerate(remaining)]
        scored.sort(key=lambda x: -x[0])
        
        # Look at top N candidates, pick the one with the best SECOND step too
        best_total = -1
        best_choice = None
        best_idx = -1
        
        for score, idx, candidate in scored[:lookahead]:
            # Simulate: if we pick this candidate, what's its best next match?
            remaining_after = [r for j, r in enumerate(remaining) if j != idx]
            if remaining_after:
                next_best = max(compatibility_score(candidate, r) for r in remaining_after)
            else:
                next_best = 0
            
            total = score + (0.5 * next_best)  # weight current step more than future
            
            if total > best_total:
                best_total = total
                best_choice = candidate
                best_idx = idx
        
        current = remaining.pop(best_idx)
        ordered.append(current)
    
    return pd.DataFrame(ordered)

In [12]:
def fix_bpm_doubling(bpm):
    """
    librosa sometimes doubles tempo for complex/orchestral tracks.
    If BPM is unrealistically high, assume it's doubled and halve it.
    """
    if bpm > 180:
        return round(bpm / 2, 1)
    return bpm

df['bpm'] = df['bpm'].apply(fix_bpm_doubling)

In [16]:
reordered_df = greedy_reorder(df)

print("Reordered playlist:")
print(reordered_df[['filename', 'genre', 'bpm', 'camelot', 'energy']].to_string(index=False))

Reordered playlist:
                                             filename     genre    bpm camelot  energy
../data/genres_original\classical\classical.00009.wav classical 117.50      6A   0.067
          ../data/genres_original\rock\rock.00004.wav      rock 103.36      6A   0.272
../data/genres_original\classical\classical.00006.wav classical  99.38      5A   0.085
          ../data/genres_original\rock\rock.00003.wav      rock  95.70      4A   0.225
      ../data/genres_original\hiphop\hiphop.00002.wav    hiphop  99.38      3A   0.184
          ../data/genres_original\rock\rock.00005.wav      rock  99.38      8A   0.269
      ../data/genres_original\hiphop\hiphop.00004.wav    hiphop  95.70      8A   0.351
          ../data/genres_original\rock\rock.00007.wav      rock  89.10      8A   0.213
          ../data/genres_original\rock\rock.00009.wav      rock  76.00      8B   0.184
../data/genres_original\classical\classical.00003.wav classical  71.78      8B   0.088
        ../data/genres_

In [17]:
def total_playlist_score(ordered_list):
    """
    Average compatibility score across all consecutive pairs.
    Higher = smoother overall playlist.
    """
    if len(ordered_list) < 2:
        return 0
    
    scores = []
    for i in range(len(ordered_list) - 1):
        s = compatibility_score(ordered_list[i], ordered_list[i+1])
        scores.append(s)
    
    return sum(scores) / len(scores)

ordered_list = reordered_df.to_dict('records')
baseline_score = total_playlist_score(ordered_list)
print(f"Greedy playlist score: {baseline_score:.2f}/100")

Greedy playlist score: 72.73/100


In [18]:
def two_opt_improve(ordered_list, max_iterations=1000):
    """
    Improves a playlist ordering by reversing segments
    whenever it increases total playlist score.
    """
    best = ordered_list.copy()
    best_score = total_playlist_score(best)
    
    n = len(best)
    improved = True
    iterations = 0
    
    while improved and iterations < max_iterations:
        improved = False
        iterations += 1
        
        # Try every pair of positions (i, j) to reverse between them
        for i in range(1, n - 1):
            for j in range(i + 1, n):
                # Create a candidate by reversing the segment [i:j+1]
                candidate = best[:i] + best[i:j+1][::-1] + best[j+1:]
                candidate_score = total_playlist_score(candidate)
                
                if candidate_score > best_score:
                    best = candidate
                    best_score = candidate_score
                    improved = True
        
    return best, best_score, iterations

improved_list, improved_score, n_iters = two_opt_improve(ordered_list)

print(f"Greedy score:        {baseline_score:.2f}/100")
print(f"After 2-opt score:   {improved_score:.2f}/100")
print(f"Improvement:         +{improved_score - baseline_score:.2f}")
print(f"Iterations until convergence: {n_iters}")

Greedy score:        72.73/100
After 2-opt score:   76.58/100
Improvement:         +3.86
Iterations until convergence: 4


In [19]:
print("Final reordered playlist (after 2-opt):")
for i, track in enumerate(improved_list):
    print(f"{i:2d}. {track['filename']:35s} {track['genre']:10s} BPM:{track['bpm']:6.1f}  {track['camelot']:4s}  E:{track['energy']:.3f}")

Final reordered playlist (after 2-opt):
 0. ../data/genres_original\classical\classical.00009.wav classical  BPM: 117.5  6A    E:0.067
 1. ../data/genres_original\blues\blues.00000.wav blues      BPM: 123.0  6A    E:0.221
 2. ../data/genres_original\rock\rock.00004.wav rock       BPM: 103.4  6A    E:0.272
 3. ../data/genres_original\classical\classical.00006.wav classical  BPM:  99.4  5A    E:0.085
 4. ../data/genres_original\rock\rock.00003.wav rock       BPM:  95.7  4A    E:0.225
 5. ../data/genres_original\hiphop\hiphop.00002.wav hiphop     BPM:  99.4  3A    E:0.184
 6. ../data/genres_original\hiphop\hiphop.00001.wav hiphop     BPM:  92.3  2A    E:0.450
 7. ../data/genres_original\hiphop\hiphop.00008.wav hiphop     BPM: 172.3  1A    E:0.306
 8. ../data/genres_original\disco\disco.00006.wav disco      BPM: 123.0  1A    E:0.336
 9. ../data/genres_original\disco\disco.00004.wav disco      BPM: 123.0  1B    E:0.289
10. ../data/genres_original\hiphop\hiphop.00007.wav hiphop     BPM: 123.